In [1]:
import zipfile
import os
import pandas as pd
import numpy as np
import torch
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
import optuna # <-- NUEVO: Importar optuna
import numpy as np
from sklearn.metrics import accuracy_score, f1_score
zip_file_path = '../data/MeIA2025-Reto-01.zip'

extracted_folder_path = '../data/extracted_corpus/'

# 2. Crear la carpeta de extracción si no existe
if not os.path.exists(extracted_folder_path):
    os.makedirs(extracted_folder_path)
    print(f"Carpeta '{extracted_folder_path}' creada.")
# 3. Descomprimir archivo  
try:
    with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
        zip_ref.extractall(extracted_folder_path)
    print(f"'{zip_file_path}' descomprimido exitosamente en '{extracted_folder_path}'.")
except FileNotFoundError:
    print(f"Error: El archivo ZIP no se encontró en '{zip_file_path}'. Verifica la ruta.")
except Exception as e:
    print(f"Ocurrió un error al descomprimir el archivo: {e}")
    
# 4. Listar los archivos descomprimidos (para verificar)
print("\nArchivos en la carpeta del corpus:")
corpus_files = os.listdir(extracted_folder_path)
for file_name in corpus_files:
    print(f"- {file_name}")

# Definir la ruta base donde se extrajo el contenido del ZIP
# Asegúrate de que esta ruta sea correcta relativa a tu notebook test.ipynb
# Si tu notebook está en 'notebooks/' y la extracción está en 'data/extracted_corpus/Datos-MelA-Reto-01/'
base_extracted_path = '../data/extracted_corpus/Datos-MeIA-Reto-01/'

# Rutas completas a los archivos XLSX
train_file_path = os.path.join(base_extracted_path, 'MeIA_2025_train.xlsx')
test_file_path = os.path.join(base_extracted_path, 'MeIA_2025_test_wo_labels.xlsx')

print(f"Intentando cargar el archivo de entrenamiento desde: {train_file_path}")
print(f"Intentando cargar el archivo de prueba desde: {test_file_path}")

try:
    # Cargar el dataset de entrenamiento
    
    df_train = pd.read_excel(train_file_path)
    print(f"Datos de entrenamiento cargados correctamente")
    # Cargar el dataset de prueba (sin etiquetas)
    df_test = pd.read_excel(test_file_path)
    print(f"Datos de test cargados correctamente")


except FileNotFoundError:
    print(f"Error: Uno de los archivos XLSX no se encontró.")
    print(f"Asegúrate de que las rutas sean correctas: '{train_file_path}' y '{test_file_path}'")
    print(f"Y que la carpeta 'Datos-MelA-Reto-01' esté dentro de 'extracted_corpus'.")
except Exception as e:
    print(f"Ocurrió un error al cargar los archivos Excel: {e}")


2025-06-16 20:01:23.267339: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1750118483.566661  106044 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1750118483.691796  106044 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1750118485.095652  106044 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1750118485.095689  106044 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1750118485.095691  106044 computation_placer.cc:177] computation placer alr

'../data/MeIA2025-Reto-01.zip' descomprimido exitosamente en '../data/extracted_corpus/'.

Archivos en la carpeta del corpus:
- Datos-MeIA-Reto-01
Intentando cargar el archivo de entrenamiento desde: ../data/extracted_corpus/Datos-MeIA-Reto-01/MeIA_2025_train.xlsx
Intentando cargar el archivo de prueba desde: ../data/extracted_corpus/Datos-MeIA-Reto-01/MeIA_2025_test_wo_labels.xlsx
Datos de entrenamiento cargados correctamente
Datos de test cargados correctamente


In [4]:
print("Creando input estructurado...")
df_train['structured_text'] = df_train.apply(
    lambda row: f"tipo: {str(row['Type']).lower()}. pueblo: {str(row['Town']).lower()}. reseña: {str(row['Review']).lower()}",
    axis=1
)

Creando input estructurado...


In [ ]:
# Paso 2: Preparar el DataFrame y convertirlo a un Dataset de Hugging Face
# Renombramos las columnas y ajustamos las etiquetas (si no lo has hecho en el df original)
df_train_beto = df_train.rename(columns={'structured_text': 'text', 'Polarity': 'label'})
df_train_beto['label'] = df_train_beto['label'].apply(lambda x: int(x) - 1)

# Convertir a Dataset
dataset = Dataset.from_pandas(df_train_beto)

# Dividir en entrenamiento y validación
train_test_split = dataset.train_test_split(test_size=0.1)
train_dataset = train_test_split['train']
eval_dataset = train_test_split['test']


# Paso 3: Cargar el tokenizador y el modelo BETO
# ¡Este es el cambio principal!
model_name = "dccuchile/bert-base-spanish-wwm-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=5) # 5 clases de polaridad

# Paso 4: Tokenizar los datos
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)

tokenized_train_dataset = train_dataset.map(tokenize_function, batched=True)
tokenized_eval_dataset = eval_dataset.map(tokenize_function, batched=True)

# Paso 5: Definir la métrica de evaluación
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    f1 = f1_score(labels, predictions, average="weighted")
    accuracy = accuracy_score(labels, predictions)
    return {"accuracy": accuracy, "f1_weighted": f1}

# Paso 6: Configurar y ejecutar el entrenamiento
training_args = TrainingArguments(
    output_dir="./results_beto",     # Nuevo directorio para no sobreescribir el anterior
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs_beto',       # Nuevo directorio de logs
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_eval_dataset,
    compute_metrics=compute_metrics,
)

# ¡Iniciar el entrenamiento!
trainer.train()

print("¡Entrenamiento con BETO completado!")

# Para guardar el modelo final y el tokenizador
# trainer.save_model("./mi_modelo_beto_final")